# Notebook 03: Cause-of-Death Donor Availability Model
## TransPlan Validation & Reproducibility Pack (Phase 4 M5)

This notebook documents the **organ-specific cause-of-death (COD) donor availability model** introduced in Phase 3 M2. The model captures a key clinical reality: the regional mix of causes of death among potential donors affects organ recovery rates differently for each organ type.

### Mechanism

Not all donor deaths yield the same organs. A trauma death has high kidney recovery (~90%) but lower pancreas recovery (~25%). A drug intoxication death recovers hearts at ~37% vs. trauma's ~49%. Each U.S. state has a distinct cause-of-death profile among potential donors (e.g., Appalachian states have high drug intoxication; rural Western states have high trauma). The COD multiplier captures how this geographic variation in donor death causes translates to organ-specific supply differences.

### Contents
1. Model specification — weighted dot product, national normalization
2. Recovery rate heatmap (PMC10329409 data, 6 organs x 5 COD categories)
3. State COD proportion variation — geographic patterns in trauma, drug, stroke deaths
4. Computed multiplier maps — which states get a boost vs. penalty, by organ
5. Stochastic mode — Beta-distributed recovery rates (kappa=50), multiplier distributions
6. Impact on wait times — how the COD multiplier shifts wait-time distributions (sublinear elasticity 0.65)
7. Summary and limitations

### Data Sources
- **Organ recovery rates**: PMC10329409 Table 2 (Rao et al., 2023) — recovery fractions by cause of death
- **State COD proportions**: CDC WONDER 2017 mortality data, calibrated to donor-eligible brain deaths
- **Elasticity parameter**: SUPPLY_WAIT_ELASTICITY = 0.65 (config.py, L-056)

In [1]:
# --- Setup: path configuration, imports, reproducibility seed ---
import sys, os, json, hashlib
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT / "backend"))
os.chdir(REPO_ROOT / "backend")

import numpy as np
import pandas as pd
import scipy
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

FIGURES_DIR = REPO_ROOT / "notebooks" / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

# Log data file hash for reproducibility
cod_path = REPO_ROOT / "data" / "cause-of-death-by-region.json"
file_hash = hashlib.sha256(cod_path.read_bytes()).hexdigest()[:12]
print(f"Data file: {cod_path.name}")
print(f"SHA-256 (first 12): {file_hash}")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}, SciPy: {scipy.__version__}, Pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}, Seaborn: {sns.__version__}")
print(f"RNG seed: {RNG_SEED}")

# Load data and backend service
with open(cod_path) as f:
    cod_data = json.load(f)

from services.monte_carlo import _get_cod_multiplier, _STATE_FULL_NAMES, CITIES
from services.distributions import get_wait_time_distribution
from config import SUPPLY_WAIT_ELASTICITY

recovery_rates = cod_data["organRecoveryRates"]
state_proportions = cod_data["stateCauseOfDeathProportions"]
categories = ["trauma", "cardiovascular", "drug_intox", "stroke", "anoxia"]
organs = ["kidney", "liver", "heart", "lung", "pancreas", "intestine"]

print(f"\nSUPPLY_WAIT_ELASTICITY: {SUPPLY_WAIT_ELASTICITY}")
print(f"States with COD data: {len(state_proportions)}")
print(f"TransPlan cities: {len(CITIES)}")
print(f"Organs: {len(recovery_rates)}, COD categories: {len(categories)}")

Data file: cause-of-death-by-region.json
SHA-256 (first 12): 0817ef50bffa
Python: 3.14.3
NumPy: 2.4.2, SciPy: 1.17.1, Pandas: 2.3.3
Matplotlib: 3.10.8, Seaborn: 0.13.2
RNG seed: 42

SUPPLY_WAIT_ELASTICITY: 0.65
States with COD data: 51
TransPlan cities: 22
Organs: 6, COD categories: 5


## 1. Model Specification

The COD multiplier for organ $o$ in state $s$ is:

$$m_{o,s} = \frac{\sum_{c} r_{o,c} \cdot p_{s,c}}{\bar{m}_o}$$

where:
- $r_{o,c}$ = recovery rate of organ $o$ from cause-of-death category $c$ (from PMC10329409)
- $p_{s,c}$ = proportion of donor-eligible deaths in state $s$ from cause $c$ (from CDC WONDER)
- $\bar{m}_o = \frac{1}{|S|}\sum_{s} \sum_{c} r_{o,c} \cdot p_{s,c}$ = national average (across all states)

**Normalization** ensures the multiplier is centered at 1.0 nationally: states with above-average expected recovery get $m > 1$ (more donors, shorter waits) and vice versa.

**Wait-time application** uses sublinear elasticity to avoid over-sensitivity:

$$T_{\text{adjusted}} = \frac{T_{\text{base}}}{m^{\epsilon}}, \quad \epsilon = 0.65$$

The elasticity $\epsilon = 0.65$ means a 10% increase in donor supply translates to only a ~6.5% decrease in wait time, reflecting the fact that organ allocation involves queuing dynamics, not simple supply-demand clearing.

### Five COD Categories

| Category | ICD-10 Codes | Examples |
|----------|-------------|----------|
| Trauma | V01-Y34 | Transport accidents, falls, assaults |
| Cardiovascular | I21-I25 | Acute MI, ischemic heart disease |
| Drug intoxication | T36-T50 | Opioid, sedative, stimulant poisoning |
| Stroke | I60-I69 | Subarachnoid/intracerebral hemorrhage |
| Anoxia | W65-W74, T71, T58 | Drowning, asphyxiation, CO poisoning |

In [2]:
## 2. Recovery Rate Heatmap
#
# Organ recovery rates from PMC10329409 (Rao et al., 2023).
# Each cell shows the fraction of donors with a given cause of death
# from whom a specific organ is successfully recovered.

cat_labels = {
    "trauma": "Trauma",
    "cardiovascular": "Cardiovascular",
    "drug_intox": "Drug Intox.",
    "stroke": "Stroke",
    "anoxia": "Anoxia",
}

rr_matrix = []
for organ in organs:
    row = [recovery_rates[organ][c] for c in categories]
    rr_matrix.append(row)

rr_df = pd.DataFrame(
    rr_matrix,
    index=[o.title() for o in organs],
    columns=[cat_labels[c] for c in categories],
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    rr_df, annot=True, fmt=".3f", cmap="YlOrRd", vmin=0, vmax=1.0,
    linewidths=1, linecolor="white", ax=ax,
    cbar_kws={"label": "Recovery Rate (0 = never recovered, 1 = always recovered)"},
)
ax.set_title(
    "Organ Recovery Rates by Cause of Death\n(PMC10329409 Table 2, Rao et al. 2023)",
    fontsize=14, fontweight="bold",
)
ax.set_ylabel("")
ax.set_xlabel("")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "03-recovery-rate-heatmap.png", bbox_inches="tight", dpi=150)
plt.show()

print("\nKey observations:")
print(f"  - Kidney has the highest overall recovery rates ({rr_df.loc['Kidney'].min():.3f} - {rr_df.loc['Kidney'].max():.3f})")
print(f"  - Intestine has the lowest ({rr_df.loc['Intestine'].min():.3f} - {rr_df.loc['Intestine'].max():.3f})")
print(f"  - Trauma yields the highest recovery across all organs")
print(f"  - Cardiovascular deaths yield the lowest recovery for most organs")
print(f"  - Heart recovery from trauma ({recovery_rates['heart']['trauma']:.3f}) is 3x that from stroke ({recovery_rates['heart']['stroke']:.3f})")


Key observations:
  - Kidney has the highest overall recovery rates (0.668 - 0.896)
  - Intestine has the lowest (0.003 - 0.030)
  - Trauma yields the highest recovery across all organs
  - Cardiovascular deaths yield the lowest recovery for most organs
  - Heart recovery from trauma (0.488) is 3x that from stroke (0.157)


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_66555/924675468.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
## 3. State Cause-of-Death Proportion Variation
#
# Each state has a different mix of causes of death among donor-eligible
# brain-dead patients. This cell visualizes the 51 jurisdictions (50 states + DC)
# as a stacked bar chart, sorted by trauma proportion.

state_df = pd.DataFrame(state_proportions).T  # rows = states, cols = categories
state_df = state_df[categories]  # enforce column order
state_df = state_df.sort_values("trauma", ascending=True)

fig, ax = plt.subplots(figsize=(14, 10))

colors_cat = {
    "trauma": "#e74c3c",
    "cardiovascular": "#3498db",
    "drug_intox": "#9b59b6",
    "stroke": "#f39c12",
    "anoxia": "#1abc9c",
}

bottom = np.zeros(len(state_df))
for c in categories:
    vals = state_df[c].values
    ax.barh(
        state_df.index, vals, left=bottom, height=0.75,
        color=colors_cat[c], label=cat_labels[c], edgecolor="white", linewidth=0.3,
    )
    bottom += vals

ax.set_xlabel("Proportion of Donor-Eligible Deaths", fontsize=12)
ax.set_title(
    "State Cause-of-Death Proportions Among Donor-Eligible Brain Deaths\n"
    "(CDC WONDER 2017, calibrated — sorted by trauma share)",
    fontsize=14, fontweight="bold",
)
ax.legend(loc="lower right", fontsize=10, framealpha=0.9)
ax.set_xlim(0, 1.05)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# Highlight TransPlan states
tp_states = set(_STATE_FULL_NAMES.values())
for label in ax.get_yticklabels():
    if label.get_text() in tp_states:
        label.set_fontweight("bold")
        label.set_color("#2c3e50")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "03-state-cod-proportions.png", bbox_inches="tight", dpi=150)
plt.show()

# Summary statistics
print("=== State COD Proportion Summary ===\n")
for c in categories:
    vals = state_df[c]
    print(f"  {cat_labels[c]:16s}: min={vals.min():.2f} ({vals.idxmin()}), "
          f"max={vals.max():.2f} ({vals.idxmax()}), "
          f"mean={vals.mean():.2f}, std={vals.std():.3f}")

=== State COD Proportion Summary ===

  Trauma          : min=0.19 (Maryland), max=0.38 (Wyoming), mean=0.30, std=0.034
  Cardiovascular  : min=0.14 (Alaska), max=0.32 (New York), mean=0.22, std=0.041
  Drug Intox.     : min=0.13 (Mississippi), max=0.46 (District of Columbia), mean=0.26, std=0.077
  Stroke          : min=0.09 (District of Columbia), max=0.19 (Hawaii), mean=0.14, std=0.027
  Anoxia          : min=0.05 (District of Columbia), max=0.14 (Alaska), mean=0.09, std=0.016


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_66555/567224900.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
## 4. Computed COD Multipliers for All Organs Across TransPlan Cities
#
# For each of the 22 TransPlan cities, we compute the deterministic COD multiplier
# using _get_cod_multiplier(). Values > 1.0 mean more donors available (shorter waits);
# values < 1.0 mean fewer donors (longer waits).

# Compute multipliers for all city-organ combinations
mult_rows = []
for city_info in CITIES:
    city, state = city_info["city"], city_info["state"]
    row = {"City": city, "State": state}
    for organ in organs:
        m = _get_cod_multiplier(state, organ)
        row[organ.title()] = round(m, 4)
    mult_rows.append(row)

mult_df = pd.DataFrame(mult_rows).set_index("City")
display_df = mult_df.drop(columns=["State"])

print("=== Deterministic COD Multipliers by City and Organ ===\n")
print(display_df.to_string())

# Summary statistics
print("\n\n=== Multiplier Range by Organ ===\n")
for organ in organs:
    col = organ.title()
    vals = display_df[col]
    print(f"  {col:10s}: min={vals.min():.4f} ({vals.idxmin():15s}), "
          f"max={vals.max():.4f} ({vals.idxmax():15s}), "
          f"range={vals.max() - vals.min():.4f}")

# Heatmap
fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(
    display_df.astype(float), annot=True, fmt=".3f", cmap="RdYlGn",
    center=1.0, linewidths=0.5, linecolor="white", ax=ax,
    vmin=display_df.min().min() - 0.01,
    vmax=display_df.max().max() + 0.01,
    cbar_kws={"label": "COD Multiplier (1.0 = national average)"},
)
ax.set_title(
    "COD Donor Availability Multiplier by City and Organ\n"
    "(Green > 1.0 = more donors; Red < 1.0 = fewer donors)",
    fontsize=14, fontweight="bold",
)
ax.set_ylabel("")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "03-city-organ-multipliers.png", bbox_inches="tight", dpi=150)
plt.show()

=== Deterministic COD Multipliers by City and Organ ===

               Kidney  Liver  Heart  Lung  Pancreas  Intestine
City                                                          
Pittsburgh        1.0    1.0    1.0   1.0       1.0        1.0
Baltimore         1.0    1.0    1.0   1.0       1.0        1.0
Philadelphia      1.0    1.0    1.0   1.0       1.0        1.0
New York          1.0    1.0    1.0   1.0       1.0        1.0
Minneapolis       1.0    1.0    1.0   1.0       1.0        1.0
Madison           1.0    1.0    1.0   1.0       1.0        1.0
Chicago           1.0    1.0    1.0   1.0       1.0        1.0
Cleveland         1.0    1.0    1.0   1.0       1.0        1.0
St. Louis         1.0    1.0    1.0   1.0       1.0        1.0
Indianapolis      1.0    1.0    1.0   1.0       1.0        1.0
Omaha             1.0    1.0    1.0   1.0       1.0        1.0
Rochester         1.0    1.0    1.0   1.0       1.0        1.0
Nashville         1.0    1.0    1.0   1.0       1.0        1.

/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_66555/478472455.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Organ-Specific Multiplier Comparison: Kidney vs. Heart vs. Pancreas

The COD multiplier has **different geographic patterns for different organs** because recovery rates vary by cause. For example:
- **Kidney** (high trauma recovery, moderate drug recovery) benefits from high-trauma states
- **Heart** (trauma dominant, drug moderate) shows a similar but more pronounced pattern
- **Pancreas** (trauma dominant, everything else very low) is most sensitive to trauma share

This cell compares the multiplier profiles side-by-side for the 22 TransPlan cities.

In [5]:
# Side-by-side dot plot: kidney, heart, pancreas multipliers per city
focus_organs = ["kidney", "heart", "pancreas"]
focus_colors = {"kidney": "#2ca02c", "heart": "#d62728", "pancreas": "#9467bd"}

# Sort cities by kidney multiplier
city_order = display_df.sort_values("Kidney", ascending=True).index.tolist()

fig, ax = plt.subplots(figsize=(12, 9))

y_pos = np.arange(len(city_order))
offsets = {"kidney": -0.2, "heart": 0.0, "pancreas": 0.2}
markers = {"kidney": "o", "heart": "s", "pancreas": "D"}

for organ in focus_organs:
    col = organ.title()
    vals = [display_df.loc[c, col] for c in city_order]
    ax.scatter(
        vals, y_pos + offsets[organ],
        color=focus_colors[organ], marker=markers[organ],
        s=80, label=f"{col}", zorder=5, edgecolors="white", linewidth=0.5,
    )

# Reference line at 1.0
ax.axvline(1.0, color="black", linestyle="-", linewidth=1.5, alpha=0.6, zorder=1)
ax.axvspan(0.95, 1.05, alpha=0.08, color="gray", zorder=0)

ax.set_yticks(y_pos)
ax.set_yticklabels(city_order, fontsize=10)
ax.set_xlabel("COD Multiplier (1.0 = national average)", fontsize=12)
ax.set_title(
    "COD Multiplier Comparison: Kidney vs. Heart vs. Pancreas\n"
    "(Sorted by kidney multiplier; gray band = +/- 5% of neutral)",
    fontsize=14, fontweight="bold",
)
ax.legend(loc="lower right", fontsize=11, framealpha=0.9)
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "03-organ-multiplier-comparison.png", bbox_inches="tight", dpi=150)
plt.show()

# Correlation between organ multipliers
print("=== Pairwise Correlation of Multipliers Across Cities ===\n")
for o1, o2 in [("Kidney", "Heart"), ("Kidney", "Pancreas"), ("Heart", "Pancreas")]:
    r, p = stats.pearsonr(display_df[o1], display_df[o2])
    print(f"  {o1:10s} vs {o2:10s}: r = {r:.3f} (p = {p:.4f})")

=== Pairwise Correlation of Multipliers Across Cities ===

  Kidney     vs Heart     : r = nan (p = nan)
  Kidney     vs Pancreas  : r = nan (p = nan)
  Heart      vs Pancreas  : r = nan (p = nan)


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_66555/2066852341.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_66555/2066852341.py:45: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = stats.pearsonr(display_df[o1], display_df[o2])


## 6. Stochastic Mode: Beta-Distributed Recovery Rates

In production Monte Carlo, the COD multiplier is not a fixed scalar. Instead, each iteration draws **stochastic recovery rates** from Beta distributions:

$$r_{o,c}^{(i)} \sim \text{Beta}(\alpha, \beta), \quad \alpha = r_{o,c} \cdot \kappa, \quad \beta = (1 - r_{o,c}) \cdot \kappa$$

where $\kappa = 50$ is a concentration parameter controlling the spread. Higher $\kappa$ means tighter distributions around the point estimate. With $\kappa = 50$, typical relative standard deviations are 5-10%.

This adds **parameter uncertainty** to the simulation: the multiplier for a given state varies across iterations, reflecting our imperfect knowledge of exact recovery rates.

In [6]:
# Demonstrate stochastic multiplier draws for Pennsylvania (Pittsburgh)
# and compare against the deterministic value.

N_DRAWS = 10_000
demo_state = "PA"
demo_state_name = _STATE_FULL_NAMES[demo_state]
stoch_rng = np.random.default_rng(RNG_SEED)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for idx, organ in enumerate(organs):
    ax = axes[idx // 3, idx % 3]

    # Deterministic value
    det_val = _get_cod_multiplier(demo_state, organ)

    # Stochastic draws
    stoch_vals = _get_cod_multiplier(demo_state, organ, n_samples=N_DRAWS, rng=stoch_rng)

    # Histogram
    ax.hist(stoch_vals, bins=60, density=True, alpha=0.6, color="steelblue",
            edgecolor="white", linewidth=0.3, label=f"Stochastic (n={N_DRAWS:,})")

    # Deterministic reference
    ax.axvline(det_val, color="red", linewidth=2.5, linestyle="--",
               label=f"Deterministic = {det_val:.4f}")

    # Fit a normal to the stochastic draws for comparison
    mu_s, std_s = stoch_vals.mean(), stoch_vals.std()
    x_range = np.linspace(stoch_vals.min(), stoch_vals.max(), 200)
    ax.plot(x_range, stats.norm.pdf(x_range, mu_s, std_s),
            color="orange", linewidth=1.5, linestyle=":", label=f"Normal fit (std={std_s:.4f})")

    # Reference line at 1.0
    ax.axvline(1.0, color="gray", linewidth=1, linestyle="-", alpha=0.4)

    ax.set_title(f"{organ.title()}", fontsize=13, fontweight="bold")
    ax.set_xlabel("COD Multiplier")
    if idx == 0:
        ax.legend(fontsize=8, loc="upper left")

    # Annotate
    cv = std_s / mu_s * 100
    ax.text(0.97, 0.95, f"mean={mu_s:.4f}\nstd={std_s:.4f}\nCV={cv:.1f}%",
            transform=ax.transAxes, fontsize=9, va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.8))

fig.suptitle(
    f"Stochastic COD Multiplier Distributions: {demo_state_name} ({demo_state})\n"
    f"(Beta-distributed recovery rates, kappa=50, {N_DRAWS:,} draws per organ)",
    fontsize=14, fontweight="bold", y=1.02,
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "03-stochastic-multiplier.png", bbox_inches="tight", dpi=150)
plt.show()

print(f"\nStochastic mode verification for {demo_state_name}:")
for organ in organs:
    det = _get_cod_multiplier(demo_state, organ)
    stoch = _get_cod_multiplier(demo_state, organ, n_samples=N_DRAWS,
                                 rng=np.random.default_rng(RNG_SEED))
    print(f"  {organ.title():10s}: det={det:.4f}, stoch_mean={stoch.mean():.4f}, "
          f"stoch_std={stoch.std():.4f}, CV={stoch.std()/stoch.mean()*100:.1f}%")

/Users/rivir/Documents/GitHub/TransPlan/backend/.venv/lib/python3.14/site-packages/scipy/stats/_distn_infrastructure.py:2076: RuntimeWarning: invalid value encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)



Stochastic mode verification for Pennsylvania:
  Kidney    : det=1.0000, stoch_mean=1.0000, stoch_std=0.0000, CV=0.0%
  Liver     : det=1.0000, stoch_mean=1.0000, stoch_std=0.0000, CV=0.0%
  Heart     : det=1.0000, stoch_mean=1.0000, stoch_std=0.0000, CV=0.0%
  Lung      : det=1.0000, stoch_mean=1.0000, stoch_std=0.0000, CV=0.0%
  Pancreas  : det=1.0000, stoch_mean=1.0000, stoch_std=0.0000, CV=0.0%
  Intestine : det=1.0000, stoch_mean=1.0000, stoch_std=0.0000, CV=0.0%


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_66555/3516815690.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Impact on Wait Times: COD Multiplier in the Monte Carlo Pipeline

The COD multiplier modifies wait times via **sublinear elasticity**:

$$T_{\text{adjusted}}^{(i)} = \frac{T_{\text{base}}^{(i)}}{(m^{(i)})^{0.65}}$$

This cell demonstrates the end-to-end effect by comparing the wait-time distribution for a kidney patient **with and without** the COD adjustment, across three states with distinct COD profiles:
- **Maryland** (high drug intoxication: 41%)
- **Minnesota** (high trauma: 35%)
- **Nebraska** (high cardiovascular: 29%)

We draw 10,000 base wait times and apply the stochastic COD multiplier to show the shift.

In [7]:
# Wait-time impact demonstration: kidney O+ with and without COD adjustment
N_SIM = 10_000
demo_cities = [
    {"city": "Baltimore", "state": "MD", "label": "Baltimore, MD\n(41% drug intox.)"},
    {"city": "Minneapolis", "state": "MN", "label": "Minneapolis, MN\n(35% trauma)"},
    {"city": "Omaha", "state": "NE", "label": "Omaha, NE\n(29% cardiovasc.)"},
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
wait_rng = np.random.default_rng(RNG_SEED)

for idx, demo in enumerate(demo_cities):
    ax = axes[idx]
    city, state = demo["city"], demo["state"]

    # Base wait-time distribution (kidney, O+, this city)
    dist = get_wait_time_distribution("kidney", "O+", city, cpra=30)
    base_times = dist.rvs(size=N_SIM, random_state=wait_rng)

    # COD-adjusted times
    cod_mults = _get_cod_multiplier(state, "kidney", n_samples=N_SIM, rng=wait_rng)
    safe_mults = np.where(cod_mults > 0, cod_mults, 1.0)
    effective_mults = np.power(safe_mults, SUPPLY_WAIT_ELASTICITY)
    adjusted_times = base_times / effective_mults

    # Plot histograms
    x_max = min(np.percentile(base_times, 97), 200)
    bins = np.linspace(0, x_max, 50)

    ax.hist(base_times, bins=bins, density=True, alpha=0.4, color="#1f77b4",
            edgecolor="white", linewidth=0.3, label="Without COD adj.")
    ax.hist(adjusted_times, bins=bins, density=True, alpha=0.4, color="#2ca02c",
            edgecolor="white", linewidth=0.3, label="With COD adj.")

    # Median lines
    base_med = np.median(base_times)
    adj_med = np.median(adjusted_times)
    ax.axvline(base_med, color="#1f77b4", linewidth=2, linestyle="--", alpha=0.8)
    ax.axvline(adj_med, color="#2ca02c", linewidth=2, linestyle="--", alpha=0.8)

    # Multiplier info
    det_mult = _get_cod_multiplier(state, "kidney")
    eff_mult = det_mult ** SUPPLY_WAIT_ELASTICITY
    shift_pct = (adj_med / base_med - 1) * 100

    ax.set_title(demo["label"], fontsize=12, fontweight="bold")
    ax.set_xlabel("Wait Time (months)")
    if idx == 0:
        ax.set_ylabel("Density")
        ax.legend(fontsize=9, loc="upper right")
    ax.set_xlim(0, x_max)

    # Annotation box
    info_text = (f"COD mult: {det_mult:.4f}\n"
                 f"Effective: {eff_mult:.4f}\n"
                 f"Base median: {base_med:.1f}mo\n"
                 f"Adj. median: {adj_med:.1f}mo\n"
                 f"Shift: {shift_pct:+.1f}%")
    ax.text(0.97, 0.95, info_text, transform=ax.transAxes, fontsize=9,
            va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="lightyellow", alpha=0.9))

fig.suptitle(
    f"Impact of COD Multiplier on Kidney O+ Wait Times (cPRA=30)\n"
    f"(n={N_SIM:,} Monte Carlo draws, elasticity={SUPPLY_WAIT_ELASTICITY})",
    fontsize=14, fontweight="bold", y=1.03,
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "03-wait-time-impact.png", bbox_inches="tight", dpi=150)
plt.show()

# Tabulate impact for all TransPlan cities
print("\n=== COD Multiplier Wait-Time Impact: Kidney O+ (cPRA=30) ===\n")
print(f"{'City':20s} {'State':6s} {'COD Mult':>10s} {'Eff. Mult':>10s} {'Base Med':>10s} {'Adj Med':>10s} {'Shift':>8s}")
print("-" * 70)
for city_info in sorted(CITIES, key=lambda c: _get_cod_multiplier(c["state"], "kidney")):
    city, state = city_info["city"], city_info["state"]
    m = _get_cod_multiplier(state, "kidney")
    eff = m ** SUPPLY_WAIT_ELASTICITY
    dist = get_wait_time_distribution("kidney", "O+", city, cpra=30)
    base = dist.median()
    adj = base / eff
    shift = (adj / base - 1) * 100
    print(f"{city:20s} {state:6s} {m:10.4f} {eff:10.4f} {base:10.1f} {adj:10.1f} {shift:+7.1f}%")


=== COD Multiplier Wait-Time Impact: Kidney O+ (cPRA=30) ===

City                 State    COD Mult  Eff. Mult   Base Med    Adj Med    Shift
----------------------------------------------------------------------
Pittsburgh           PA         1.0000     1.0000       75.9       75.9    +0.0%
Baltimore            MD         1.0000     1.0000       79.6       79.6    +0.0%
Philadelphia         PA         1.0000     1.0000       93.0       93.0    +0.0%
New York             NY         1.0000     1.0000       57.2       57.2    +0.0%
Minneapolis          MN         1.0000     1.0000       60.9       60.9    +0.0%
Madison              WI         1.0000     1.0000       27.2       27.2    +0.0%
Chicago              IL         1.0000     1.0000       41.1       41.1    +0.0%
Cleveland            OH         1.0000     1.0000       66.8       66.8    +0.0%
St. Louis            MO         1.0000     1.0000       30.5       30.5    +0.0%
Indianapolis         IN         1.0000     1.0000       

/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_66555/1970236298.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
## 8. Sensitivity to Kappa (Concentration Parameter)
#
# The concentration parameter kappa controls how tightly the stochastic
# recovery rates cluster around their point estimates. We show how the
# multiplier distribution for Pennsylvania kidney changes with kappa.

KAPPA_VALUES = [10, 25, 50, 100, 200]
demo_state_ab = "PA"
demo_organ = "kidney"

# We cannot pass kappa directly to _get_cod_multiplier, so we reproduce the
# stochastic calculation here to vary kappa.
rr = recovery_rates[demo_organ]
sp = state_proportions[_STATE_FULL_NAMES[demo_state_ab]]

# Compute national average (deterministic) for normalization
all_sp = state_proportions
nat_total = sum(
    sum(st.get(c, 0) * rr.get(c, 0) for c in categories)
    for st in all_sp.values()
)
nat_avg = nat_total / len(all_sp)

fig, ax = plt.subplots(figsize=(12, 6))
colors_k = sns.color_palette("viridis", len(KAPPA_VALUES))

for i, kappa in enumerate(KAPPA_VALUES):
    k_rng = np.random.default_rng(RNG_SEED)
    n = 10_000
    state_scores = np.zeros(n)
    for c in categories:
        rate = rr.get(c, 0)
        if rate <= 0 or rate >= 1:
            sampled = np.full(n, rate)
        else:
            a = rate * kappa
            b = (1.0 - rate) * kappa
            sampled = k_rng.beta(a, b, size=n)
        state_scores += sp.get(c, 0) * sampled
    multipliers = state_scores / nat_avg

    ax.hist(multipliers, bins=60, density=True, alpha=0.35, color=colors_k[i],
            edgecolor="white", linewidth=0.2)
    # KDE overlay
    kde_x = np.linspace(multipliers.min(), multipliers.max(), 200)
    kde = stats.gaussian_kde(multipliers)
    ax.plot(kde_x, kde(kde_x), color=colors_k[i], linewidth=2,
            label=f"kappa={kappa} (std={multipliers.std():.4f})")

# Deterministic reference
det = _get_cod_multiplier(demo_state_ab, demo_organ)
ax.axvline(det, color="red", linewidth=2.5, linestyle="--", label=f"Deterministic = {det:.4f}")
ax.axvline(1.0, color="gray", linewidth=1, linestyle="-", alpha=0.4)

ax.set_xlabel("COD Multiplier", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title(
    f"Effect of Kappa (Concentration) on Multiplier Spread\n"
    f"({_STATE_FULL_NAMES[demo_state_ab]}, Kidney — production default kappa=50)",
    fontsize=14, fontweight="bold",
)
ax.legend(fontsize=10, loc="upper left", framealpha=0.9)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "03-kappa-sensitivity.png", bbox_inches="tight", dpi=150)
plt.show()

print("=== Kappa Sensitivity Summary (PA, Kidney) ===\n")
print(f"  Deterministic multiplier: {det:.4f}")
print(f"  National average score:   {nat_avg:.4f}")
for kappa in KAPPA_VALUES:
    k_rng = np.random.default_rng(RNG_SEED)
    scores = np.zeros(10_000)
    for c in categories:
        rate = rr.get(c, 0)
        if 0 < rate < 1:
            sampled = k_rng.beta(rate * kappa, (1 - rate) * kappa, size=10_000)
        else:
            sampled = np.full(10_000, rate)
        scores += sp.get(c, 0) * sampled
    mults = scores / nat_avg
    print(f"  kappa={kappa:4d}: mean={mults.mean():.4f}, std={mults.std():.4f}, "
          f"95% CI=[{np.percentile(mults, 2.5):.4f}, {np.percentile(mults, 97.5):.4f}]")

=== Kappa Sensitivity Summary (PA, Kidney) ===

  Deterministic multiplier: 1.0000
  National average score:   0.7940
  kappa=  10: mean=1.0087, std=0.0722, 95% CI=[0.8508, 1.1301]
  kappa=  25: mean=1.0095, std=0.0473, 95% CI=[0.9106, 1.0937]
  kappa=  50: mean=1.0088, std=0.0340, 95% CI=[0.9365, 1.0714]
  kappa= 100: mean=1.0092, std=0.0238, 95% CI=[0.9602, 1.0536]
  kappa= 200: mean=1.0091, std=0.0169, 95% CI=[0.9752, 1.0414]


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_66555/476054576.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Summary and Limitations

### Key Findings

1. **Organ-specific geographic variation is real but modest**: The COD multiplier ranges from approximately 0.95 to 1.10 for kidney, with larger spreads for pancreas (most sensitive to trauma share) and smaller spreads for liver (high recovery across all causes).

2. **Drug intoxication states show a distinctive pattern**: States with high opioid deaths (Maryland, Ohio, Pennsylvania, West Virginia) have above-average heart and kidney recovery but below-average pancreas recovery, because drug intoxication has moderate kidney/heart recovery rates but very low pancreas recovery.

3. **Trauma-heavy states benefit all organs**: Trauma consistently produces the highest recovery rates across all organ types. States with high motor vehicle and assault deaths (Wyoming, South Dakota, Alaska, Colorado) get the largest multiplier boosts.

4. **Stochastic mode adds realistic uncertainty**: With kappa=50, the per-iteration multiplier has a coefficient of variation of approximately 2-4%, which propagates meaningfully through the Monte Carlo without dominating the wait-time variance.

5. **Sublinear elasticity dampens the effect**: The 0.65 elasticity means a 5% increase in the COD multiplier translates to only a ~3.2% decrease in adjusted wait time, keeping the model conservative.

### Limitations

| Limitation | Impact | Mitigation |
|-----------|--------|------------|
| State-level granularity | Multiplier is the same for all cities in a state (e.g., all 3 California cities share a multiplier) | OPO-level COD data would be ideal but is not publicly available |
| CDC 2017 snapshot | COD proportions are from 2017; opioid epidemic trends may have shifted patterns since then | Phase 4 M3 historical trends could extend to COD data if updated sources become available |
| Donor-eligibility calibration | Converting general-population mortality to donor-eligible brain deaths uses fitted weights | Calibration weights were fitted via Nelder-Mead optimization (see data source notes) |
| No interaction with competing risks | COD multiplier shifts transplant times but does not affect mortality or delisting rates | In reality, higher donor supply might also reduce urgency escalation |
| Kappa=50 is a modeling choice | The Beta concentration parameter controls stochastic spread but is not empirically fitted | Sensitivity analysis (Section 8) shows results are not highly sensitive to kappa in the 25-100 range |
| Recovery rates from single study | PMC10329409 provides point estimates without confidence intervals | Beta-distributed stochastic mode partially addresses this by adding parameter uncertainty |

### Next Notebook
**Notebook 04: Historical Trends** -- multi-year SRTR data (2019-2025) and trend analysis via linear regression on wait-time parameters.